# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S03: Trực Giác Mạng Tích Chập (CNN): Convolution, Stride & Pooling

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S03
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook lập trình trực quan hóa phép tích chập và các tầng CNN
* **Mức độ liên quan VOAI / Olympic AI:** Nền tảng kiến trúc cốt lõi để làm chủ mọi mô hình Thị giác máy tính thi đấu.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu tại sao mạng MLP truyền thống thất bại thảm hại khi xử lý ảnh (Bùng nổ tham số & Phá hủy cấu trúc không gian 2D).
- Nắm vững cơ chế toán học của phép Tích chập 2D (Convolution): Kernel/Filter trượt trên ma trận ảnh.
- Làm chủ 3 siêu tham số cốt tử của tầng tích chập: Kernel Size, Padding (Same vs Valid), và Stride.
- Hiểu vai trò của Tầng gộp cực đại (MaxPool2d): Giảm độ phân giải không gian và tạo tính bất biến dịch chuyển (Translation Invariance).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S01 (Tensors hình ảnh) và M3_S03 (nn.Module).
- Hiểu phép nhân tích vô hướng của hai ma trận nhỏ.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Hãy thử làm một phép tính: Nếu bạn đưa một bức ảnh màu có kích thước vừa phải $256 	imes 256 	imes 3$ vào một tầng MLP có 512 nơ-ron, số lượng trọng số cần học chỉ riêng cho tầng đầu tiên sẽ là: $256 	imes 256 	imes 3 	imes 512 pprox \mathbf{100	ext{ triệu tham số}}$! Mạng nơ-ron sẽ nổ tung bộ nhớ, đồng thời việc duỗi phẳng (Flatten) bức ảnh thành vector 1D sẽ phá hủy hoàn toàn mối quan hệ không gian giữa các pixel liền kề.

**Mạng Tích Chập (Convolutional Neural Networks - CNN)** ra đời như một cuộc cách mạng: Thay vì kết nối tất cả các pixel, CNN sử dụng các bộ lọc nhỏ (Kernels, ví dụ $3 	imes 3$) trượt trên bức ảnh, chia sẻ trọng số (Weight Sharing) để phát hiện từ các nét cơ bản (đường cong, góc nhọn) đến các bộ phận phức tạp (mắt, mũi, bánh xe). Hiểu rõ CNN là điều kiện tiên quyết của mọi chuyên gia AI!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Cơ chế Tích chập 2D (Convolution Operation)
* Một bộ lọc (Filter/Kernel, ví dụ ma trận $3 \times 3$) trượt qua từng vùng của bức ảnh.
* Tại mỗi vị trí, nó thực hiện phép nhân từng phần tử tương ứng rồi cộng tổng lại (Element-wise multiplication & sum) để tạo ra một giá trị trong **Bản đồ đặc trưng (Feature Map)**.
* **Chia sẻ trọng số (Weight Sharing):** Cùng một bộ lọc được dùng để quét toàn bộ bức ảnh, giúp giảm hàng triệu tham số so với MLP.

### 2. Các siêu tham số cốt lõi của `nn.Conv2d`
* **Kernel Size ($K$):** Kích thước cửa sổ trượt (phổ biến nhất là $3 \times 3$).
* **Stride ($S$):** Bước nhảy của cửa sổ trượt (Stride = 1: trượt từng pixel; Stride = 2: nhảy cóc, giảm kích thước ảnh đi một nửa).
* **Padding ($P$):** Đệm thêm các vòng số 0 xung quanh viền ảnh.
  - *Valid Padding ($P=0$):* Không đệm, ảnh đầu ra bị co nhỏ lại.
  - *Same Padding ($P = \frac{K - 1}{2}$ khi $S=1$):* Giữ nguyên kích thước chiều dài và chiều rộng của ảnh!
* **Công thức kích thước đầu ra:**
  $$H_{out} = \left\lfloor \frac{H_{in} - K + 2P}{S} \right\rfloor + 1$$

### 3. Tầng Gộp Cực Đại (MaxPool2d)
`nn.MaxPool2d(kernel_size=2, stride=2)` lấy giá trị lớn nhất trong mỗi cửa sổ $2 \times 2$:
* Giảm kích thước không gian đi một nửa (giảm 75% lượng tính toán cho các tầng sau).
* Giúp mô hình có tính bất biến với các xê dịch nhỏ của vật thể.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cơ chế trượt của Bộ lọc Tích chập (Kernel 3x3):
Vùng ảnh gốc (3x3):       Bộ lọc Sobel (3x3):      Kết quả Feature Map:
   [ 10,  20,  10 ]           [ -1,  0,  1 ]
   [ 10,  20,  10 ]     *     [ -2,  0,  2 ]   ==> [ Giá trị biên độ cạnh ]
   [ 10,  20,  10 ]           [ -1,  0,  1 ]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# 1. Cai dat phep tich chap thu cong bang NumPy de phat hien canh doc (Vertical Edge Detection)
image = np.zeros((10, 10))
image[:, 5:] = 10.0 # Nua trai la 0 (den), nua phai la 10 (trang) -> Co 1 canh doc o giua

# Bo loc phat hien canh doc (Vertical Sobel-like Kernel)
kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
])

# Tinh toan phep truot convolution thu cong
output_manual = np.zeros((8, 8))
for i in range(8):
    for j in range(8):
        window = image[i:i+3, j:j+3]
        output_manual[i, j] = np.sum(window * kernel)

print("===== MA TRAN ANH GOC (10x10) =====")
print(image[:4, :])
print("\n===== FEATURE MAP PHAT HIEN CANH (8x8) =====")
print(output_manual[:4, :])
print("-> Cot co gia tri cao vuot troi chinh la vi tri cua canh bien!")

### 🔍 Code Walkthrough (Khối 1)
Khối code lập trình tích chập thủ công bằng NumPy:
- Minh họa rõ ràng cơ chế trượt cửa sổ $3 	imes 3$ và tính tổng tích từng phần tử.
- Cho thấy trực quan cách bộ lọc trích xuất đường biên thẳng đứng khi chuyển giao giữa vùng tối và vùng sáng.


In [ ]:
# 2. Xay dung tang Conv2d va MaxPool2d bang PyTorch
conv_layer = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1, stride=1)
pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)

# Gia lap mot batch gom 2 anh xam kich thuoc 28x28 (giong anh MNIST)
dummy_images = torch.randn(2, 1, 28, 28)

# Chay qua Conv2d
feature_maps = conv_layer(dummy_images)
print(f"Kich thuoc sau Conv2d (Padding=1 giu nguyen H, W): {feature_maps.shape}")

# Chay qua MaxPool2d
pooled_maps = pool_layer(feature_maps)
print(f"Kich thuoc sau MaxPool2d (Giam mot nua do phan giai): {pooled_maps.shape}")

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa tầng `nn.Conv2d` và `nn.MaxPool2d` trong PyTorch:
- Đầu vào: `[2, 1, 28, 28]` (Batch 2 ảnh, 1 kênh xám, $28 	imes 28$).
- Qua Conv2d (`padding=1`): Giữ nguyên kích thước không gian $(28, 28)$ nhưng mở rộng lên 4 bản đồ đặc trưng (`[2, 4, 28, 28]`).
- Qua MaxPool2d (`stride=2`): Thu nhỏ kích thước còn một nửa $(14, 14)$ (`[2, 4, 14, 14]`).


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Tính toán kích thước đầu ra
Cho một ảnh đầu vào có kích thước $[1, 3, 64, 64]$. Hãy tính kích thước đầu ra bằng công thức toán học sau khi đi qua tầng:
`nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=2, padding=2)`.
Kiểm tra lại bằng cách khởi tạo tầng và truyền tensor giả lập qua.

### Bài 2 (Easy): Đếm số tham số của tầng Tích chập
Tính toán số lượng tham số cần học (Weights + Biases) của tầng `nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)`. So sánh con số này với một tầng `nn.Linear(3*32*32, 32)`.

### Bài 3 (Medium): Trực quan hóa Feature Maps
Sử dụng Matplotlib vẽ 4 bản đồ đặc trưng (Feature Maps) được sinh ra từ biến `feature_maps[0]` ở đoạn code trên để quan sát các góc nhìn khác nhau mà mạng nơ-ron học được.

### Bài 4 (Medium): So sánh MaxPool2d vs AvgPool2d
Tìm hiểu tầng gộp trung bình `nn.AvgPool2d(kernel_size=2, stride=2)`. Nêu sự khác biệt trực giác: Tại sao MaxPool thường được ưa chuộng hơn cho việc giữ lại các tín hiệu biên sắc nét?

### Bài 5 (Challenge): Cài đặt Depthwise Separable Convolution
Tìm hiểu cấu trúc Depthwise Separable Convolution (nền tảng của kiến trúc MobileNet dành cho thiết bị di động). Sử dụng PyTorch với tham số `groups` trong `nn.Conv2d` để xây dựng cấu trúc này và chứng minh nó giảm hơn 80% số lượng phép tính so với tích chập tiêu chuẩn.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong thi đấu Olympic AI, hiểu rõ công thức kích thước đầu ra của CNN là vũ khí tối thượng giúp bạn tránh lỗi `RuntimeError: mat1 and mat2 shapes cannot be multiplied` khi kết nối tầng Convolution cuối cùng vào tầng `nn.Linear` (Flatten layer). Thí sinh chuyên nghiệp luôn biết chính xác sau bao nhiêu lần MaxPool thì kích thước không gian ảnh bị thu nhỏ lại bao nhiêu để khai báo đúng kích thước cho tầng Fully Connected.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh áp dụng quá nhiều tầng MaxPool liên tiếp trên ảnh có kích thước nhỏ (ví dụ ảnh $16 	imes 16$), khiến kích thước không gian bị giảm về 0:

```python
# CODE CÓ LỖI THU NHỎ QUÁ ĐÀ:
# Anh 16x16 -> Pool 1: 8x8 -> Pool 2: 4x4 -> Pool 3: 2x2 -> Pool 4: 1x1 -> Pool 5: 0x0 (Error: Kernel size can't be greater than input size!)
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao số lượng tầng Downsampling (MaxPool hoặc Stride 2) bị giới hạn bởi độ phân giải của ảnh đầu vào.
2. Nêu cách tính số lần gộp tối đa có thể áp dụng cho một bức ảnh kích thước $W 	imes H$.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Ưu điểm vượt trội nhất của Mạng Tích Chập (CNN) so với Mạng Perceptron Đa Tầng (MLP) khi xử lý dữ liệu hình ảnh là gì?
   - A. CNN không cần dùng hàm mất mát.
   - B. CNN bảo toàn cấu trúc không gian 2D của ảnh và chia sẻ trọng số (Weight Sharing), giúp giảm mạnh số lượng tham số và tăng khả năng nhận diện bất biến vị trí.
   - C. CNN chỉ chạy được trên CPU.
   - D. CNN luôn có độ chính xác 100%.

2. Một bộ lọc tích chập kích thước $3 	imes 3$ trên một ảnh màu có 3 kênh (RGB) chứa bao nhiêu trọng số (chưa tính bias)?
   - A. 9
   - B. 27 ($3 	imes 3 	imes 3$)
   - C. 81
   - D. 3

3. Thiết lập tham số nào trong `nn.Conv2d` với `kernel_size=3` và `stride=1` sẽ giữ nguyên kích thước dài và rộng của ảnh đầu vào (Same Padding)?
   - A. `padding=0`
   - B. `padding=1`
   - C. `padding=2`
   - D. `padding=3`

4. Tầng `nn.MaxPool2d(kernel_size=2, stride=2)` thực hiện thao tác gì trên bản đồ đặc trưng?
   - A. Lấy giá trị lớn nhất trong mỗi cửa sổ $2 	imes 2$, giúp giảm kích thước chiều dài và chiều rộng đi một nửa.
   - B. Nhân đôi độ phân giải của bức ảnh.
   - C. Tính giá trị trung bình của toàn bộ bức ảnh.
   - D. Xóa bỏ các giá trị âm.

5. Trong một tầng CNN, khái niệm "Feature Map" (Bản đồ đặc trưng) đại diện cho điều gì?
   - A. Bản đồ địa lý vị trí máy tính đang đặt.
   - B. Ma trận đầu ra thể hiện mức độ phản hồi của một bộ lọc (Kernel) tại từng vị trí không gian trên bức ảnh.
   - C. Nhãn thực tế của bài toán.
   - D. Bảng thống kê số lượng tham số.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Lý do mạng MLP truyền thống bất lực trước dữ liệu hình ảnh.
  - Bản chất toán học của phép Tích chập 2D và nguyên lý chia sẻ trọng số.
  - 3 siêu tham số cốt lõi: Kernel Size, Stride, Padding và công thức kích thước đầu ra.
  - Vai trò giảm chiều và trích xuất đặc trưng sắc nét của MaxPool2d.
* **What I can now do:**
  - Tự tay cài đặt và tính toán kích thước tensor qua các khối tích chập.
  - Phối hợp nhịp nhàng giữa `nn.Conv2d` và `nn.MaxPool2d`.
* **What I still need to practice:**
  - Lắp ráp thành một kiến trúc mạng phân loại ảnh hoàn chỉnh ở Buổi 4.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S04**, chúng ta sẽ chính thức ráp nối toàn bộ khối sức mạnh: **Xây Dựng End-to-End Image Classification Pipeline với PyTorch CNN** — Tự tay lập trình mạng CNN phân loại ảnh hoàn chỉnh, viết training loop cho dữ liệu ảnh và phân tích ma trận nhầm lẫn!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Trực giác mạng tích chập (CNN) & Weight sharing ✓
- Cơ chế Convolution 2D, Kernel, Stride & Padding ✓
- Tầng gộp MaxPool2d & Feature maps visualization ✓

* **Session Deliverable:** Mã nguồn cài đặt phép tích chập và theo dõi kích thước Feature Maps
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
